In [ ]:
from pathlib import Path

import httpx
import os
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root() -> Path:
    search_paths: list[Path] = []

    try:
        notebook_file = get_ipython().user_ns.get("__vsc_ipynb_file__")  # type: ignore[name-defined]
        if notebook_file:
            search_paths.append(Path(notebook_file).resolve().parent.parent)
    except Exception:
        pass

    if Path.cwd().name == "RawLLMCall":
        search_paths.append(Path.cwd().parent)

    search_paths.extend([Path.cwd(), *Path.cwd().parents])

    seen: set[Path] = set()
    for directory in search_paths:
        resolved = directory.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / "pyproject.toml").is_file():
            return resolved

    raise FileNotFoundError(
        "Could not find project root. Open the langchain-training folder in Cursor, then re-run."
    )


project_root = find_project_root()
load_dotenv(project_root / ".env", override=True)

provider_url = os.getenv("PROVIDER_URL", "").strip()
provider_key = os.getenv("PROVIDER_KEY", "").strip()
if not provider_url or not provider_key:
    raise ValueError(
        f"Set PROVIDER_URL and PROVIDER_KEY in {project_root / '.env'}"
    )

client = OpenAI(
    api_key=provider_key,
    base_url=provider_url,
    http_client=httpx.Client(trust_env=False, timeout=60.0),
)

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {
            "role": "user",
            "content": "Say Hello"
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
print(os.getenv("PROVIDER_URL"))